# Boolean predicates, PyXAI, and image regions

Start here. A neural reader produces scores; thresholding gives the bits used by the actual classifier. PyXAI explains that classifier. We separately track which pixels the readers may use.

The named bird rule below is illustrative, not a biological rule or a CUB result. Run this notebook from the `ltn_pyxai_starter` directory with the root requirements installed.


In [1]:
import itertools
import json
import numpy as np
import torch
from boolean_cbm import BooleanHead, compile_boolean_function, fit_tree, threshold_scores
from boolean_cbm.predicates import CropPredicateReader, masked_concept_loss, region_union, replace_outside


## 1. Define the deployed Boolean function

`red_crown AND (long_beak OR NOT dark_wing)` is an actual classifier here. No surrogate is fitted. The converter enumerates all eight Boolean inputs and builds an equivalent PyXAI tree. Use this route only for small heads (at most ten concepts by default).


In [2]:
names = ['red_crown', 'long_beak', 'dark_wing']
def rule(c):
    return int(c[0] and (c[1] or not c[2]))

head = compile_boolean_function(rule, names)
q = np.array([0.9, 0.8, 0.1])  # illustrative reader outputs
c = threshold_scores(q)
print('bits:', c.tolist(), 'prediction:', head.predict(c))
reasons = head.reasons(c, all_reasons=True)
for r in reasons:
    print(r, head.describe(r), 'verified:', head.verify(r, c))


bits: [1, 1, 0] prediction: 1
(1, -3) ['red_crown', 'NOT dark_wing'] verified: True
(1, 2) ['red_crown', 'long_beak'] verified: True


## 2. See the PyXAI call and a counterexample

Positive IDs denote predicted true predicates; negative IDs denote predicted false predicates. Neither proves that the image annotation is correct. For every returned reason, our independent verifier searches all compatible tree paths and checks subset minimality.

PyXAI 2.0.1's `is_sufficient_reason` uses sampling; we do not use it as the exact check. `is_implicant` is an exact tree entailment query.


In [3]:
from pyxai import Explaining
explainer = head.pyxai_explainer()
explainer.set_instance(c.tolist())
print('PyXAI reasons:', explainer.sufficient_reason(n=Explaining.ALL))
for r in reasons:
    print(r, 'PyXAI entailment:', explainer.is_implicant(r))

# Crown alone is insufficient: a completion can reverse the class.
witness = head.counterexample((1,), target=head.predict(c))
print('Counterexample to crown alone:', witness, 'class:', head.predict(witness))


PyXAI reasons: ((1, 2), (1, -3))
(1, -3) PyXAI entailment: True
(1, 2) PyXAI entailment: True
Counterexample to crown alone: (1, 0, 1) class: 0


## 3. Map both positive and negative literals to their full support

Boxes below are a synthetic layout, with `(y0,y1,x0,x1)` coordinates and exclusive upper bounds. Their union certifies pixel sufficiency only when the readers truly depend on these crops. A box over a globally encoded DINO feature map is not such a guarantee.


In [4]:
boxes = np.array([[0,4,0,4], [0,4,4,8], [4,8,0,4]])
for r in reasons:
    mask = region_union(boxes, r, (8,8))
    print(head.describe(r), 'union area:', float(mask.mean()))

from boolean_cbm.demo import run
print(json.dumps(run(), indent=2))


['red_crown', 'NOT dark_wing'] union area: 0.5
['red_crown', 'long_beak'] union area: 0.5
{
  "synthetic_only": true,
  "formula": "c1 AND (c2 OR NOT c3)",
  "bits": [
    1,
    1,
    0
  ],
  "prediction": 1,
  "certificates": [
    {
      "reason": [
        1,
        -3
      ],
      "named_literals": [
        "red_in_patch_1",
        "NOT blue_in_patch_3"
      ],
      "area_fraction": 0.5,
      "outside_edits_checked": 25
    },
    {
      "reason": [
        1,
        2
      ],
      "named_literals": [
        "red_in_patch_1",
        "green_in_patch_2"
      ],
      "area_fraction": 0.5,
      "outside_edits_checked": 25
    }
  ],
  "boolean_worlds_checked": 8,
  "reasons_checked": 10,
  "scope": "fixed boxes and deterministic crop readers; no claim of semantic correctness"
}


## 4. Trainable crop-first predicates

This small reader slices before convolution. Replace its per-crop network with a stronger crop encoder only while preserving that dependency contract. The example performs one supervised optimization step; it does not establish semantic quality.

A missing label is excluded using `observed`. It is never a negative target. At deployment, finite scores are thresholded; an unobserved or occluded concept is not thereby semantically verified.


In [5]:
torch.manual_seed(7)
reader = CropPredicateReader(n_predicates=3)
optimizer = torch.optim.Adam(reader.parameters(), lr=1e-3)
images = torch.rand(2, 3, 8, 8)
fixed_boxes = np.broadcast_to(boxes, (2,3,4)).copy()
labels = torch.tensor([[1., 0., float('nan')], [0., 1., 1.]])
observed = torch.isfinite(labels)
optimizer.zero_grad()
loss = masked_concept_loss(reader(images, fixed_boxes), labels, observed)
loss.backward()
optimizer.step()
reader.eval()
with torch.no_grad():
    scores = reader(images, fixed_boxes).sigmoid().numpy()
print('Masked training loss:', float(loss.detach()))
print('Deployed bits:', threshold_scores(scores).tolist())


Masked training loss: 0.6741202473640442
Deployed bits: [[1, 0, 1], [1, 0, 1]]


## 5. Train a tree as the actual head

For an immediately usable PyXAI baseline on many CUB predicates, fit the deployed task tree directly to frozen predicted bits and task labels. This structural export does not enumerate the whole Boolean cube. It is not an approximation to another task head.

The tiny dataset below is only an interface demonstration. For a real evaluation, use held-out images, tune on validation only, and report reader accuracy separately.


In [6]:
train_bits = np.array(list(itertools.product((0,1), repeat=3)))
train_y = np.array([rule(row) for row in train_bits])
learned = fit_tree(train_bits, train_y, names, max_depth=3, min_samples_leaf=1)
print('Prediction:', learned.predict(c))
print('Checked explanation:', learned.describe(learned.reasons(c)[0]))


Prediction: 1
Checked explanation: ['red_crown', 'long_beak']


## 6. Connect your saved CUB scores

Create an NPZ containing `q[N,M]`, `y[N]`, `split[N]`, `ids[N]`, `groups[N]`, and `concept_names[M]`. Use the frozen reader's probabilities for `q`, and task labels for `y`. Every edit of one original image must share its group and split. This interface rejects cross-split groups.

```bash
python -m boolean_cbm.train_tree --data cub_predicates.npz --out results/cub_tree
```

This writes `head.json`, `metrics.json`, and `explanations.jsonl`. Reload without labels:

```python
head, thresholds = BooleanHead.load('results/cub_tree/head.json')
c = threshold_scores(q_new, thresholds)
reason = head.reasons(c)[0]
```

For your existing small difflogic circuit, pass its hard `cvpr_logic.verify.predict` function to `compile_boolean_function`. Above ten concepts, keep the existing circuit SAT verifier. No claim of spatial certification can be made from cached probability arrays alone.

See `boolean_cbm/README.md` for the exact scope, primary documentation, and integration details.
